<a href="https://colab.research.google.com/github/mahathi-kannan/enterprise-data-analytics-pipeline/blob/main/04-geospatial-expansion/geospatial_expansion_modeling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Install mapping utility packages
!pip install folium

import pandas as pd
import numpy as np
import folium

In [3]:
# Load the IEA global dataset
df = pd.read_csv('IEA Global EV Data 2024.csv')

# Clean column white-spaces defensively
df.columns = df.columns.str.strip()

# Target Filter: Historical EV Sales for passenger cars in the latest full baseline year (2023)
market_df = df[
    (df['category'] == 'Historical') &
    (df['parameter'] == 'EV sales') &
    (df['mode'] == 'Cars') &
    (df['year'] == 2023)
]

# Aggregate sales volume per country/region (sum up BEV and PHEV powertrains)
country_sales = market_df.groupby('region')['value'].sum().reset_index()

# Filter out non-country regional aggregates to prevent double-counting numbers
aggregates_to_drop = ['World', 'Europe', 'EU27', 'Rest of the world']
country_sales = country_sales[~country_sales['region'].isin(aggregates_to_drop)]

# Rename columns professionally
country_sales.columns = ['Country', 'Total_EV_Sales_2023']

print("--- Filtered Country Matrix Preview ---")
print(country_sales.sort_values(by='Total_EV_Sales_2023', ascending=False).head(10))

--- Filtered Country Matrix Preview ---
           Country  Total_EV_Sales_2023
7            China            8100520.0
48             USA            1393000.0
19         Germany             700260.0
18          France             470310.0
50  United Kingdom             450025.0
33     Netherlands             210065.0
2          Belgium             193009.0
5           Canada             171013.0
45          Sweden             171002.0
27           Japan             140320.0


In [4]:
# Precise coordinate mapping for countries inside the IEA dataset
country_gps = {
    'China': [35.8617, 104.1954], 'USA': [37.0902, -95.7129], 'Germany': [51.1657, 10.4515],
    'France': [46.2276, 2.2137], 'United Kingdom': [55.3781, -3.4360], 'Netherlands': [52.1326, 5.2913],
    'Belgium': [50.5039, 4.4699], 'Canada': [56.1304, -106.3468], 'Sweden': [60.1282, 18.6435],
    'Japan': [36.2048, 138.2529], 'Korea': [35.9078, 127.7669], 'Italy': [41.8719, 12.5674],
    'Spain': [40.4637, -3.7492], 'Norway': [60.4720, 8.4689], 'Australia': [-25.2744, 133.7751],
    'India': [20.5937, 78.9629], 'Denmark': [56.2639, 9.5018], 'Switzerland': [46.8182, 8.2275],
    'Turkiye': [38.9637, 35.2433], 'Austria': [47.5162, 14.5501], 'Israel': [31.0461, 34.8516],
    'Portugal': [39.3999, -8.2245], 'Brazil': [-14.2350, -51.9253], 'Finland': [61.9241, 25.7482],
    'New Zealand': [-40.9006, 174.8860], 'Ireland': [53.4129, -8.2439], 'Poland': [51.9194, 19.1451],
    'United Arab Emirates': [23.4241, 53.8478], 'Greece': [39.0742, 21.8243], 'Luxembourg': [49.8153, 6.1296],
    'Mexico': [23.6345, -102.5528], 'Romania': [45.9432, 24.9668], 'Iceland': [64.9631, -19.0208],
    'Czech Republic': [49.8175, 15.4730], 'Hungary': [47.1625, 19.5033], 'Colombia': [4.5709, -74.2973],
    'Slovakia': [48.6690, 19.6990], 'Slovenia': [46.1512, 14.9955], 'Costa Rica': [9.7489, -83.7534],
    'Croatia': [45.1000, 15.2000], 'Bulgaria': [42.7339, 25.4858], 'Lithuania': [55.1694, 23.8813],
    'Latvia': [56.8796, 24.6032], 'Estonia': [58.5953, 25.0136], 'Cyprus': [35.1264, 33.4299],
    'South Africa': [-30.5595, 22.9375], 'Chile': [-35.6751, -71.5430], 'Seychelles': [-4.6796, 55.4920]
}

# Attach latitude and longitude to our aggregated sales summary table
country_sales['Latitude'] = country_sales['Country'].map(lambda x: country_gps.get(x, [np.nan, np.nan])[0])
country_sales['Longitude'] = country_sales['Country'].map(lambda x: country_gps.get(x, [np.nan, np.nan])[1])

# Drop any entries that lack valid map coordinates
country_sales = country_sales.dropna(subset=['Latitude', 'Longitude'])
print("Geospatial coordination mapping verified!")

Geospatial coordination mapping verified!


In [5]:
# Initialize a global base map map centered over neutral coordinates
global_expansion_map = folium.Map(location=[25, 10], zoom_start=2, tiles='CartoDB positron')

# Iterate and dynamically overlay data markers proportional to country sales scale
for idx, row in country_sales.iterrows():
    name = row['Country']
    sales = row['Total_EV_Sales_2023']
    lat = row['Latitude']
    lon = row['Longitude']

    # Apply natural mathematical logging scale to prevent mega-markets like China
    # from blowing up the physical map overlay size
    calculated_radius = max(int(np.log1p(sales) * 1.8), 4)

    folium.CircleMarker(
        location=[lat, lon],
        radius=calculated_radius,
        popup=f"<strong>Market:</strong> {name}<br><strong>2023 EV Sales:</strong> {int(sales):,} units",
        color='#1b4d3e',
        fill=True,
        fill_color='#2e8b57',
        fill_opacity=0.6,
        weight=1.5
    ).add_to(global_expansion_map)

# Render and inspect your global expansion visualizer
global_expansion_map